# Coffee Matchmaker — Modeling

Ruba's cleaned data still has three gaps that hurt the recommender:

| Gap | Coffees affected | What this notebook does |
|---|---|---|
| No tasting notes | 42 of 569 | **Flavor model** (multi-label classification) predicts their flavor families |
| No body, acidity or bitterness | 397, 497 and 548 of 569 | **Body model** tested first, then **taste scores** built from coffee knowledge and checked against the coffees that do state them |

At the end I save `data/coffee_enriched.csv` (real values + estimated ones, with a flag on every estimated value) and the flavor model in `models/`. The recommender uses these files.

## 1. Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 2. Load the Data

I load Ruba's clean file for modeling, and the raw file because it still has two things the clean file dropped: which altitudes were real, and the acidity text.

In [2]:
df = pd.read_csv("../data/saudi_specialty_coffee_beans_clean.csv", encoding="utf-8-sig")
raw = pd.read_csv("../data/saudi_specialty_coffee_beans.csv", encoding="utf-8-sig")

print("Clean:", df.shape)
print("Raw:  ", raw.shape)
print("Same coffees in the same order:", (df["bean_id"].values == raw["bean_id"].values).all())

Clean: (569, 40)
Raw:   (569, 54)
Same coffees in the same order: True


## 3. Fix Three Things Before Modeling

**3.1 Coffees with no tasting notes.** Ruba filled their flavor flags (`note_fruity`, `note_chocolate`...) with 0. For these 42 coffees, 0 means "unknown", not "not fruity". If I train on them, the model learns that they are not fruity, which is wrong. So I mark them with `has_notes = 0` and keep them out of the flavor model's training. They are exactly the coffees the flavor model will predict.

In [3]:
df["has_notes"] = (df["n_tasting_notes"] > 0).astype(int)
print(df["has_notes"].value_counts())

has_notes
1    527
0     42
Name: count, dtype: int64


**3.2 Altitudes that were filled in.** Ruba filled missing altitudes with the median of the same country, which is a good choice, but there is no column saying which ones were filled. I add `altitude_was_filled` from the raw file, so the model can tell a real altitude from an estimate.

In [4]:
df["altitude_was_filled"] = raw["altitude_mean_m"].isna().astype(int)
print("Filled altitudes:", df["altitude_was_filled"].sum(), "of", len(df))

Filled altitudes: 207 of 569


**3.3 Acidity text.** Ruba removed `acidity` because it is 87% empty, which is right for a model feature. I bring it back as a side column only, to check my acidity estimate in section 7. It is never used as a feature.

In [5]:
df["acidity_raw"] = raw["acidity"]
print("Coffees that state their acidity:", df["acidity_raw"].notna().sum())

Coffees that state their acidity: 72


## 4. Features

These are the coffee details a buyer can see on the product page, and that most coffees have:

- **Origin:** continent and country. Countries with fewer than 10 coffees are grouped into `Other`, because one-hot columns with 2 or 3 coffees only add noise.
- **Process** (Natural, Washed, Anaerobic...), **product type**, and **is_infused**.
- **Altitude** and `altitude_was_filled`.
- **Price per 100 g** and `price_was_filled`.

What I leave out, on purpose:

- **Roaster:** I want the model to work for coffees from roasters it has never seen, and roaster names would let it memorize each roaster's style instead of learning about coffee.
- **Roast level:** 78% Unknown.
- **Region and variety:** too many different values for 569 coffees.

In [6]:
country_counts = df["origin_country"].value_counts()
common_countries = country_counts[country_counts >= 10].index
df["origin_group"] = df["origin_country"].where(df["origin_country"].isin(common_countries), "Other")
print(df["origin_group"].value_counts())

categorical = ["origin_continent", "origin_group", "process", "product_type"]
numeric = ["is_infused", "altitude_mean_m", "altitude_was_filled", "price_per_100g_sar", "price_was_filled"]

X_base = pd.get_dummies(df[categorical], dtype=int).join(df[numeric])
print()
print("Feature columns:", X_base.shape[1])

origin_group
Colombia       151
Ethiopia       133
Multiple        60
Other           52
Brazil          48
Uganda          28
El Salvador     27
Costa Rica      24
Yemen           20
Indonesia       15
Guatemala       11
Name: count, dtype: int64

Feature columns: 37


## 5. How I Evaluate: GroupKFold by Roaster

Each roaster writes its notes and descriptions in its own style. With a normal random split, coffees from the same roaster land in both train and test, and the model gets a high score by recognizing the roaster. That score would be fake.

So I use **GroupKFold with roaster as the group**: in every fold, the test roasters are completely unseen. The score shows how the model does on a new roaster, which is the real use case.

Every model is also compared with a **Dummy baseline** that ignores the features. A model is only useful if it clearly beats it.

In [7]:
groups = df["roaster"]
cv = GroupKFold(n_splits=5)

def get_models():
    return {
        "Dummy (baseline)": DummyClassifier(strategy="prior"),
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=42),
        "Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, class_weight="balanced", random_state=42),
    }

## 6. Flavor Model (Multi-label Classification)

A coffee can be fruity **and** chocolatey at the same time, so this is **multi-label**: one yes/no model per flavor family.

- **Training rows:** the 527 coffees that have tasting notes.
- **Targets:** the six flavor families. I drop `spice` and `tea` because they appear in only 3–4% of coffees, too few to learn from.
- **Features:** only `X_base`. The tasting notes themselves are **not** features, because the flavor flags were made from them, so using them would be leakage.
- **Metric: ROC-AUC.** It measures how well the model ranks coffees that have a flavor above coffees that don't. It is not fooled by imbalance (fruity is 74% yes, citrus is 17%). 0.5 = random guessing, 1.0 = perfect.

In [8]:
flavors = ["note_fruity", "note_citrus", "note_floral", "note_chocolate", "note_nutty", "note_caramel_sweet"]

train = df["has_notes"] == 1
X_flavor = X_base[train]
groups_flavor = groups[train]

print("Training coffees:", train.sum())
print()
print("How common each flavor is:")
print(df.loc[train, flavors].mean().round(2))

Training coffees: 527

How common each flavor is:
note_fruity           0.74
note_citrus           0.17
note_floral           0.18
note_chocolate        0.41
note_nutty            0.24
note_caramel_sweet    0.37
dtype: float64


In [9]:
flavor_results = pd.DataFrame(index=flavors)

for name, model in get_models().items():
    aucs = []
    for flavor in flavors:
        y = df.loc[train, flavor]
        fold_aucs = []
        for train_idx, test_idx in cv.split(X_flavor, y, groups_flavor):
            model.fit(X_flavor.iloc[train_idx], y.iloc[train_idx])
            proba = model.predict_proba(X_flavor.iloc[test_idx])[:, 1]
            fold_aucs.append(roc_auc_score(y.iloc[test_idx], proba))
        aucs.append(np.mean(fold_aucs))
    flavor_results[name] = aucs

flavor_results.loc["Average"] = flavor_results.mean()
print("ROC-AUC on unseen roasters (average of the 5 folds):")
print(flavor_results.round(2))

ROC-AUC on unseen roasters (average of the 5 folds):
                    Dummy (baseline)  Logistic Regression  Random Forest  Gradient Boosting
note_fruity                      0.5                 0.82           0.85               0.86
note_citrus                      0.5                 0.49           0.52               0.48
note_floral                      0.5                 0.70           0.65               0.62
note_chocolate                   0.5                 0.70           0.69               0.68
note_nutty                       0.5                 0.75           0.80               0.78
note_caramel_sweet               0.5                 0.53           0.54               0.54
Average                          0.5                 0.67           0.68               0.66


In [10]:
model_names = [m for m in flavor_results.columns if m != "Dummy (baseline)"]
best_flavor_model = flavor_results.loc["Average", model_names].idxmax()
print("Best model on average:", best_flavor_model)

reliable_flavors = [f for f in flavors if flavor_results.loc[f, best_flavor_model] >= 0.65]
print("Flavors it predicts reliably (AUC >= 0.65):", reliable_flavors)
print("Flavors it cannot predict:", [f for f in flavors if f not in reliable_flavors])

Best model on average: Random Forest
Flavors it predicts reliably (AUC >= 0.65): ['note_fruity', 'note_floral', 'note_chocolate', 'note_nutty']
Flavors it cannot predict: ['note_citrus', 'note_caramel_sweet']


**Reading the results:**

- All three real models clearly beat the baseline (0.5), and they are close to each other. **Random Forest** has the best average (0.68), so I use it.
- The model is good at **fruity (0.85)** and **nutty (0.80)**: origin and process really do shape these. For example, 92% of Ethiopian coffees are fruity but only 16% of Brazilian ones, and 81% of Natural-process coffees vs 55% of Washed.
- It is no better than guessing for **citrus** and **caramel/sweet** (about 0.5). These seem to depend on the roaster's wording more than on the coffee's details.

**Why a 0.65 cut-off:** below it, the model is only a little better than guessing, and showing users a guessed flavor would be misleading. Flavors under the cut-off are left as unknown for the 42 coffees, instead of guessed.

In [11]:
flavor_models = {}
for flavor in reliable_flavors:
    model = get_models()[best_flavor_model]
    model.fit(X_flavor, df.loc[train, flavor])
    flavor_models[flavor] = model

missing = df["has_notes"] == 0
df["tasting_notes_are_predicted"] = missing.astype(int)

for flavor, model in flavor_models.items():
    proba = model.predict_proba(X_base[missing])[:, 1]
    df.loc[missing, flavor] = (proba >= 0.5).astype(int)
    df.loc[missing, flavor + "_proba"] = proba.round(2)

print("Predicted flavors for the first 5 coffees without notes:")
print(df.loc[missing, ["coffee_name"] + reliable_flavors].head())

Predicted flavors for the first 5 coffees without notes:
                              coffee_name  note_fruity  note_floral  note_chocolate  note_nutty
16          3Bean Saudi Coffee Beans 250g            0            0               1           1
57                           Chinese Crop            1            0               0           0
75             Qasr Al-Murabba Blend 250g            0            1               1           1
88  Black Knight Chaffee El Salvador 250G            1            0               1           0
89              Black Knight Minas Brazil            0            0               1           1


## 7. Body Model (Multi-class Classification)

- **Training rows:** the 172 coffees that state their body.
- **Target:** Light / Medium / Full. `Medium-Full` has only 2 coffees, so I merge it into `Full`.
- **Features:** `X_base` **plus** the flavor flags. Unlike in the flavor model, the notes are fine here: they are known for almost every coffee whose body we want to predict, and flavor tells us something about body (chocolatey coffees tend to feel heavier).
- **Metric: F1-macro.** It averages the F1 of the three classes equally, so the model can't score well by ignoring the small `Light` class (only 17 coffees). Accuracy is shown too, but it rewards always guessing `Full`.

In [12]:
df["body_target"] = df["body_level"].replace({"Medium-Full": "Full"})
known_body = df["body_target"] != "Unknown"

X_body = X_base.join(df[flavors + ["has_notes"]])
y_body = df.loc[known_body, "body_target"]

print(y_body.value_counts())

body_target
Full      94
Medium    61
Light     17
Name: count, dtype: int64


In [13]:
body_results = {}
body_predictions = {}

for name, model in get_models().items():
    pred = cross_val_predict(model, X_body[known_body], y_body, cv=cv, groups=groups[known_body])
    body_predictions[name] = pred
    body_results[name] = {
        "F1-macro": f1_score(y_body, pred, average="macro"),
        "Accuracy": accuracy_score(y_body, pred),
    }

body_results = pd.DataFrame(body_results).T
print("Body model on unseen roasters:")
print(body_results.round(2))

Body model on unseen roasters:
                     F1-macro  Accuracy
Dummy (baseline)         0.24      0.55
Logistic Regression      0.37      0.44
Random Forest            0.34      0.42
Gradient Boosting        0.40      0.49


In [14]:
best_body_model = body_results.drop("Dummy (baseline)")["F1-macro"].idxmax()
print("Best model:", best_body_model)
print()
print("Where it gets confused (rows = true body, columns = predicted):")
print(pd.crosstab(y_body, body_predictions[best_body_model], rownames=["true"], colnames=["predicted"]))

Best model: Gradient Boosting

Where it gets confused (rows = true body, columns = predicted):
predicted  Full  Light  Medium
true                          
Full         57      8      29
Light         8      3       6
Medium       25     11      25


**Honest reading:** Gradient Boosting has the best F1-macro (0.40 vs 0.24 for the baseline), so it beats pure guessing. But its accuracy (0.49) is lower than the baseline's (0.55), and it gets only 3 of the 17 `Light` coffees right.

**Decision:** I keep this section as an experiment and report it, but **I don't use these predictions in the app**. They are too close to guessing. Section 8 builds a better-founded estimate for body, acidity and bitterness.

## 8. Taste Scores: Acidity, Body and Bitterness

Users will ask things like "ابغى محصول مرارته عالية" or "حموضته خفيفة", so the app needs these three attributes for every coffee. But very few coffees state them:

| Attribute | Coffees that state it | Roasters |
|---|---|---|
| Acidity | 72 | 29 |
| Body | 172 | 49 |
| Bitterness | 21 | **1** (Roasting House only) |

That's too few labels to train a model (I test this below to be sure). Instead, I build a **taste score** for each attribute from things coffee tasters know affect it, using details most coffees have:

| Signal | Acidity | Body | Bitterness |
|---|---|---|---|
| Bright notes (lemon, berries, cherry, apple, tea...) | ⬆ | ⬇ | ⬇ |
| Deep notes (chocolate, cocoa, nuts, caramel, spices...) | ⬇ | ⬆ | ⬆ |
| Darker roast (roast level, or "dark", "Turkish", "Saudi" in the name) | ⬇ | ⬆ | ⬆ |
| Espresso-only coffees (usually roasted darker) | ⬇ | ⬆ | ⬆ |
| Filter-only coffees (usually roasted lighter) | ⬆ | | |
| African origin | ⬆ | | |
| Low-acidity origins (Brazil, Indonesia, India, Uganda) and blends | ⬇ | ⬆ | ⬆ |
| Higher altitude | ⬆ | | |
| Natural / anaerobic process | | ⬆ | |

The score is not a measurement. Its job is to **rank** coffees: when a user asks for low acidity, the coffees with the lowest acidity score come first.

In [15]:
notes_text = df["tasting_notes"].where(df["tasting_notes"] != "Unknown", "").str.lower()
name_text = (df["coffee_name"] + " " + df["other_attributes"]).str.lower()

def contains_any(texts, words):
    result = []
    for t in texts:
        found = 0
        for w in words:
            if w in t:
                found = 1
                break
        result.append(found)
    return pd.Series(result, index=texts.index)

bright_notes = contains_any(notes_text, ["lemon", "lime", "grapefruit", "citrus", "orange", "tangerine", "apple",
                                         "berr", "cherry", "pomegranate", "kiwi", "pineapple", "passion", "hibiscus",
                                         "winey", "juicy", "currant", "tea", "bergamot", "acid"])
deep_notes = contains_any(notes_text, ["chocolate", "cocoa", "roasted", "nut", "almond", "peanut", "hazelnut", "toffee",
                                       "molasses", "brown sugar", "tobacco", "smok", "spice", "cinnamon", "clove",
                                       "cardamom", "earthy", "wood"])
bitter_notes = contains_any(notes_text, ["dark chocolate", "cocoa", "roasted", "smok", "tobacco", "spice", "clove",
                                         "cardamom", "earthy", "wood", "molasses", "bitter"])
dark_in_name = contains_any(name_text, ["dark", "espresso blend", "turkish", "saudi", "arabic coffee", "french"])
light_in_name = contains_any(name_text, ["light", "filter", "omni"])

espresso_only = ((df["suits_espresso"] == 1) & (df["suits_filter"] == 0)).astype(int)
filter_only = ((df["suits_filter"] == 1) & (df["suits_espresso"] == 0)).astype(int)
african = (df["origin_continent"] == "Africa").astype(int)
low_acid_origin = df["origin_country"].isin(["Brazil", "Indonesia", "India", "Uganda", "Multiple"]).astype(int)
blend = (df["product_type"] != "single_origin").astype(int)
natural_type = df["process"].isin(["Natural", "Anaerobic", "Mixed"]).astype(int)
altitude = ((df["altitude_mean_m"] - 1800) / 300).clip(-2, 2)
roast = df["roast_level"].map({"Light": -1, "Medium-Light": -0.5, "Medium": 0, "Omni": 0,
                               "Medium-Dark": 0.5, "Dark": 1, "Saudi style": 1}).fillna(0)

print("Coffees with bright notes:", bright_notes.sum())
print("Coffees with deep notes:  ", deep_notes.sum())
print("Coffees with a dark-roast hint in the name:", dark_in_name.sum())

Coffees with bright notes: 324
Coffees with deep notes:   288
Coffees with a dark-roast hint in the name: 41


In [16]:
acidity_score = (1.0 * bright_notes + 0.5 * df["note_floral"] + 0.5 * african + 0.3 * altitude + 0.5 * filter_only
                 - 1.0 * deep_notes - 0.5 * espresso_only - 0.5 * low_acid_origin - 1.0 * roast
                 - 0.7 * dark_in_name + 0.5 * light_in_name)

body_score = (1.0 * deep_notes + 0.5 * df["note_chocolate"] + 0.5 * espresso_only + 0.5 * low_acid_origin
              + 0.5 * blend + 0.3 * natural_type + 0.7 * roast + 0.5 * dark_in_name
              - 0.5 * bright_notes - 0.5 * df["note_tea"] - 0.5 * df["note_floral"] - 0.3 * light_in_name)

bitterness_score = (1.0 * bitter_notes + 0.5 * df["note_chocolate"] + 0.7 * espresso_only + 0.5 * low_acid_origin
                    + 0.5 * blend + 1.0 * roast + 1.0 * dark_in_name
                    - 0.7 * bright_notes - 0.5 * df["note_floral"] - 0.5 * light_in_name)

print(pd.DataFrame({"acidity": acidity_score, "body": body_score, "bitterness": bitterness_score}).describe().round(2))

       acidity    body  bitterness
count   569.00  569.00      569.00
mean      0.31    0.68        0.06
std       1.35    1.18        1.22
min      -4.30   -1.80       -2.20
25%      -0.62   -0.50       -0.70
50%       0.50    0.50       -0.20
75%       1.25    1.50        0.80
max       3.50    4.00        4.50


### Checking the scores against the coffees that state these attributes

I turn every stated value into a number: Low / Light = 1, Medium = 2, High / Full = 3. Roasting House gives percentages, so ≤30% = 1, 31–59% = 2, ≥60% = 3.

Then I measure the **Spearman correlation** between the score and the stated value. It answers: *when the score is higher, is the stated value higher too?* 0 = no relation, 1 = perfect order. The **p-value** says how likely the result is pure chance (below 0.05 = unlikely to be chance).

To be sure a model can't do better, I also train a **Ridge regression** on the same signals, tested with GroupKFold by roaster like before.

In [17]:
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge

def text_to_level(text):
    if pd.isna(text):
        return None
    t = str(text).lower()
    if "%" in t:
        value = float(t.replace("%", ""))
        if value <= 30:
            return "Low"
        if value < 60:
            return "Medium"
        return "High"
    for word in ["bright", "high", "lemon", "citrus", "واضحة", "مشرقة", "فاكهية", "فواكه"]:
        if word in t:
            return "High"
    for word in ["smooth", "خفيفة", "ناعمة", "ناعمه", "كريمية"]:
        if word in t:
            return "Low"
    return "Medium"

level_number = {"Low": 1, "Light": 1, "Medium": 2, "High": 3, "Full": 3}

df["acidity_stated"] = df["acidity_raw"].apply(text_to_level)
df["bitterness_stated"] = raw["bitterness"].apply(text_to_level)
df["body_stated"] = df["body_target"].where(df["body_target"] != "Unknown")

signals = pd.DataFrame({"bright": bright_notes, "deep": deep_notes, "bitter": bitter_notes, "floral": df["note_floral"],
                        "chocolate": df["note_chocolate"], "tea": df["note_tea"], "african": african,
                        "altitude": altitude, "filter_only": filter_only, "espresso_only": espresso_only,
                        "low_acid_origin": low_acid_origin, "blend": blend, "natural": natural_type, "roast": roast,
                        "dark_name": dark_in_name, "light_name": light_in_name})

check_rows = []
for attribute, score in [("acidity", acidity_score), ("body", body_score), ("bitterness", bitterness_score)]:
    stated = df[attribute + "_stated"].map(level_number)
    has = stated.notna()
    rho, p = spearmanr(score[has], stated[has])
    row = {"attribute": attribute, "coffees": int(has.sum()), "roasters": df.loc[has, "roaster"].nunique(),
           "score: Spearman": round(rho, 2), "score: p-value": round(p, 3)}
    if df.loc[has, "roaster"].nunique() >= 5:
        pred = cross_val_predict(Ridge(alpha=3), signals[has], stated[has], cv=cv, groups=groups[has])
        row["Ridge model: Spearman"] = round(spearmanr(pred, stated[has])[0], 2)
    else:
        row["Ridge model: Spearman"] = "can't test (1 roaster)"
    check_rows.append(row)

print(pd.DataFrame(check_rows).to_string(index=False))

 attribute  coffees  roasters  score: Spearman  score: p-value  Ridge model: Spearman
   acidity       72        29             0.26           0.029                  -0.16
      body      172        48             0.25           0.001                   0.11
bitterness       21         1             0.12           0.590 can't test (1 roaster)


**Reading the results:**

- **Acidity (0.26, p = 0.03) and body (0.25, p = 0.001):** the scores rank coffees in the right direction, and it's unlikely to be chance. The relation is weak, so a score can't say "this coffee is exactly Medium", but it can put lower-acidity coffees before higher-acidity ones, which is what the recommender needs.
- **The Ridge model does worse** (−0.16 and 0.11): with this few labels, learning the weights from data fails, while weights from coffee knowledge work better. This is the reason I chose the score over a model.
- **Bitterness can't be properly tested:** only 21 coffees from a single roaster state it, and most of them say 20–30%, so there's almost nothing to rank. The result there (0.12, p = 0.59) is neither evidence for nor against the score. It's built the same way as the other two, and the app always labels it as an estimate.
- **Why the relation is weak:** roasters describe acidity and body by feel, each in their own words, so the labels themselves are noisy. Any method hits this limit with this data.
- **To be fair:** I picked the signals from coffee knowledge but looked at the data before settling the weights, and 72 coffees is a small check. The real accuracy may be a bit lower. Daniyah's blind taste test is the stronger check.

### Final values for the app

For every coffee and attribute:

1. **If the roaster states it**, I use the stated value (`..._is_estimated = 0`).
2. **Otherwise**, I use the score: the lowest third of coffees = Low / Light, the middle third = Medium, the top third = High / Full (`..._is_estimated = 1`).

I also save each score as a **percentile from 0 to 1** (`acidity_score`, `body_score`, `bitterness_score`), so the recommender can rank coffees inside the same level.

In [18]:
def make_levels(score, stated, names):
    percentile = score.rank(pct=True).round(3)
    estimated_level = pd.cut(percentile, bins=[0, 1/3, 2/3, 1], labels=names, include_lowest=True).astype(str)
    final = stated.copy()
    is_estimated = stated.isna().astype(int)
    final[stated.isna()] = estimated_level[stated.isna()]
    return final, is_estimated, percentile

df["acidity_level"], df["acidity_is_estimated"], df["acidity_score"] = make_levels(
    acidity_score, df["acidity_stated"], ["Low", "Medium", "High"])
df["body_level_final"], df["body_is_estimated"], df["body_score"] = make_levels(
    body_score, df["body_stated"], ["Light", "Medium", "Full"])
df["bitterness_level"], df["bitterness_is_estimated"], df["bitterness_score"] = make_levels(
    bitterness_score, df["bitterness_stated"], ["Low", "Medium", "High"])

for col, flag in [("acidity_level", "acidity_is_estimated"), ("body_level_final", "body_is_estimated"),
                  ("bitterness_level", "bitterness_is_estimated")]:
    print(col, "| real:", int((df[flag] == 0).sum()), "| estimated:", int(df[flag].sum()))
    print(df[col].value_counts().to_dict())
    print()

acidity_level | real: 72 | estimated: 497
{'Medium': 207, 'Low': 181, 'High': 181}

body_level_final | real: 172 | estimated: 397
{'Full': 222, 'Medium': 185, 'Light': 162}

bitterness_level | real: 21 | estimated: 548
{'Low': 239, 'High': 184, 'Medium': 146}



### Sanity check: do the extremes make sense?

Numbers aside, the real test for the demo is whether the results look right to someone who knows coffee. Here are the coffees each score ranks at the very top and bottom:

In [19]:
show = ["coffee_name", "origin_country", "process", "tasting_notes"]
in_stock = df[df["listed_out_of_stock"] == 0]

print("LOWEST acidity score:")
print(in_stock.sort_values("acidity_score").head(5)[show].to_string(index=False))
print()
print("HIGHEST acidity score:")
print(in_stock.sort_values("acidity_score", ascending=False).head(5)[show].to_string(index=False))
print()
print("HIGHEST bitterness score:")
print(in_stock.sort_values("bitterness_score", ascending=False).head(5)[show].to_string(index=False))
print()
print("LIGHTEST body score:")
print(in_stock.sort_values("body_score").head(5)[show].to_string(index=False))

LOWEST acidity score:
                           coffee_name origin_country   process                             tasting_notes
                     Sul de Minas Dark         Brazil   Natural                   Classic flavors; Almond
     Half World Saudi Style Roast 250g           Peru    Washed                      Roasted nuts; Peanut
                  Antioquia Decaf Dark       Colombia    Washed                 Dark chocolate; Dried fig
                 Tiramisu - Dark Roast       Multiple     Mixed          Dark chocolate; Cocoa; Full body
Silverskin Wafer Brazil Anaerobic 250g         Brazil Anaerobic Crispy wafer; Chocolate; Toasted hazelnut

HIGHEST acidity score:
                               coffee_name origin_country   process                                                          tasting_notes
Guji Uraga - Ethiopia - Filter Coffee 250g       Ethiopia Anaerobic Blueberry; Jasmine; Pineapple; Peach; Strawberry; Lime; Dark chocolate
 Guji Siko - Ethiopia - Filter Coffee 25

**Reading the sanity check:** the extremes match what a coffee expert would expect:

- **Lowest acidity:** dark roasts, a Saudi-style roast and Brazilian coffees, with almond, roasted nuts and dark chocolate notes.
- **Highest acidity:** Ethiopian coffees (Guji, Yirgacheffe, Gotiti) with lime, citrus, jasmine and berries.
- **Highest bitterness:** dark roasts and espresso blends with cocoa and dark chocolate. One even says "Moderate bitterness" in its notes.
- **Lightest body:** washed coffees with tea, hibiscus and jasmine notes.

This is the strongest sign the scores will give sensible results in the demo. The middle of the ranking is less certain than the extremes, which is fine for a recommender: users asking for "low acidity" get the coffees at the far end first.

## 9. Save the Enriched Data and the Model

In [20]:
enriched = df.drop(columns=["body_target", "acidity_raw", "acidity_stated", "bitterness_stated", "body_stated", "body_level"])
enriched = enriched.rename(columns={"body_level_final": "body_level"})
enriched.to_csv("../data/coffee_enriched.csv", index=False, encoding="utf-8-sig")
print("Saved data/coffee_enriched.csv:", enriched.shape)

joblib.dump({"models": flavor_models, "columns": list(X_base.columns), "model_name": best_flavor_model},
            "../models/flavor_model.joblib")
print("Saved models/flavor_model.joblib")

Saved data/coffee_enriched.csv: (569, 56)
Saved models/flavor_model.joblib


The flavor model is saved together with its column list, because a new coffee must be turned into exactly the same columns, in the same order, before the model can predict it. The body model isn't saved, since the app doesn't use it.

## Summary for Daniyah (evaluation) and the presentation

| Part | Method | Result on unseen roasters | Used in the app? |
|---|---|---|---|
| Flavor | Random Forest, multi-label | ROC-AUC: fruity 0.85, nutty 0.80, chocolate 0.69, floral 0.65 | ✅ for the 42 coffees without notes |
| Body | Gradient Boosting | F1-macro 0.40 vs 0.24 baseline, but accuracy below baseline | ❌ replaced by the body score |
| Acidity, body, bitterness | Coffee-knowledge taste scores | Spearman 0.26 (acidity), 0.25 (body); bitterness untestable | ✅ for ranking, labeled as estimates |

**Next:** the recommender (`app/recommender.py`) built on `coffee_enriched.csv`.